<a href="https://colab.research.google.com/github/HinnaNayab/DeepLearning/blob/Applied-AI/Tutorial_6_ResNet_and_VGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input as pre_process_vgg
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_process_resnet
import numpy as np
import tensorflow as tf

In [ ]:
def load_and_preprocess_image(img_path, model_name):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    if model_name == 'vgg':
        return pre_process_vgg(img_array)
    elif model_name == 'resnet':
        return pre_process_resnet(img_array)
    else:
        raise ValueError("Invalid model name. Must be 'vgg' or 'resnet'.")


In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50
vgg_model = VGG16(weights='imagenet')
resnet_model = ResNet50(weights='imagenet')


In [ ]:
from tensorflow.keras.applications.vgg16 import decode_predictions as decode_vgg
from tensorflow.keras.applications.resnet50 import decode_predictions as decode_resnet

def make_prediction(model, img_array, model_name):
  preds = model.predict(img_array)
  if model_name == 'vgg':
    return decode_vgg(preds, top=5)
  elif model_name == 'resnet':
    return decode_resnet(preds, top=5)

def print_top_5_predictions(model, img_path, model_name):
  img_array = load_and_preprocess_image(img_path, model_name)
  predictions = make_prediction(model, img_array, model_name)

  print(f"Top 5 predictions for {model_name.upper()} model:")
  for i, pred in enumerate(predictions[0]):
    print(f"{i+1}. {pred[1]} ({pred[2]*100:.2f}% probability")


In [ ]:
img_path = "/content/raccoon.jpg"

In [ ]:
print_top_5_predictions(vgg_model, img_path, 'vgg')
print()
print_top_5_predictions(resnet_model, img_path, 'resnet')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Top 5 predictions for VGG model:
1. lesser_panda (27.51% probability
2. kit_fox (12.95% probability
3. Pomeranian (11.63% probability
4. black-footed_ferret (9.61% probability
5. red_fox (7.75% probability

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Top 5 predictions for RESNET model:
1. cup (14.01% probability
2. Pomeranian (13.92% probability
3. lesser_panda (10.91% probability
4. bucket (6.98% probability
5. gibbon (3.88% probability


In [ ]:
#For Task of Tutorial 6

import os
import glob
import random
import time
import zipfile
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import (
    alexnet, AlexNet_Weights,
    resnet101, ResNet101_Weights,
    mobilenet_v2, MobileNet_V2_Weights
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Unzip the local dataset only if it is not already present
if not os.path.exists("/content/raccoon_dataset"):
    os.makedirs("/content/raccoon_dataset", exist_ok=True)
    zip_path = "/content/raccoon-classification-dataset.zip"
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("/content/raccoon_dataset")
        print("Successfully extracted dataset.")
    else:
        print(f"Warning: {zip_path} not found. Please upload the file.")

image_paths = sorted(
    glob.glob("/content/raccoon_dataset/**/*.jpg", recursive=True) +
    glob.glob("/content/raccoon_dataset/**/*.jpeg", recursive=True) +
    glob.glob("/content/raccoon_dataset/**/*.png", recursive=True)
)

print("Images found:", len(image_paths))

Device: cpu
Successfully extracted dataset.
Images found: 843


In [ ]:

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Split raccoon images into training and held-out test sets
train_paths, test_paths = train_test_split(
    image_paths,
    test_size=0.20,
    random_state=42
)

class RaccoonDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        img = Image.open(self.paths[index]).convert("RGB")
        return transform(img)

train_loader = DataLoader(
    RaccoonDataset(train_paths),
    batch_size=16,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    RaccoonDataset(test_paths),
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Training raccoon images:", len(train_paths))
print("Held-out raccoon images:", len(test_paths))

Training raccoon images: 674
Held-out raccoon images: 169


In [ ]:

def build_feature_extractor(model_name):
    if model_name == "AlexNet":
        model = alexnet(weights=AlexNet_Weights.DEFAULT)

        # Keep pretrained feature layers and hidden classifier layers
        model.classifier = nn.Sequential(
            *list(model.classifier.children())[:-1]
        )

    elif model_name == "ResNet101":
        model = resnet101(weights=ResNet101_Weights.DEFAULT)
        model.fc = nn.Identity()

    elif model_name == "MobileNetV2":
        model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        model.classifier = nn.Identity()

    model = model.to(device)
    model.eval()

    # Freeze all parameters: we only extract pretrained features
    for param in model.parameters():
        param.requires_grad = False

    return model

In [ ]:

def extract_features(model, loader):
    all_features = []

    with torch.no_grad():
        for images in loader:
            images = images.to(device)
            features = model(images)
            features = features.flatten(start_dim=1)
            all_features.append(features.cpu().numpy())

    return np.concatenate(all_features, axis=0)


def run_one_class_experiment(model_name):
    start_time = time.time()

    model = build_feature_extractor(model_name)

    train_features = extract_features(model, train_loader)
    test_features = extract_features(model, test_loader)

    # Standardize features using training data only
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)
    test_scaled = scaler.transform(test_features)

    # Learn the pattern of the raccoon training images
    detector = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.1
    )
    detector.fit(train_scaled)

    # +1 means similar to the learned raccoon pattern
    # -1 means an outlier relative to that pattern
    predictions = detector.predict(test_scaled)

    accepted = np.sum(predictions == 1)
    acceptance_rate = 100 * accepted / len(predictions)

    elapsed = time.time() - start_time

    print(f"\nArchitecture: {model_name}")
    print("Training feature shape:", train_features.shape)
    print("Held-out raccoons accepted:", accepted, "/", len(predictions))
    print(f"Raccoon acceptance rate: {acceptance_rate:.2f}%")
    print(f"Time: {elapsed:.1f} seconds")

    return {
        "Architecture": model_name,
        "Raccoon acceptance rate (%)": round(acceptance_rate, 2),
        "Time (seconds)": round(elapsed, 1)
    }

In [ ]:

results = []

for model_name in ["AlexNet", "ResNet101", "MobileNetV2"]:
    results.append(run_one_class_experiment(model_name))

results_df = pd.DataFrame(results)
display(results_df.sort_values(
    "Raccoon acceptance rate (%)",
    ascending=False
))

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:03<00:00, 76.9MB/s]



Architecture: AlexNet
Training feature shape: (674, 4096)
Held-out raccoons accepted: 125 / 169
Raccoon acceptance rate: 73.96%
Time: 42.1 seconds
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:02<00:00, 65.0MB/s]



Architecture: ResNet101
Training feature shape: (674, 2048)
Held-out raccoons accepted: 132 / 169
Raccoon acceptance rate: 78.11%
Time: 377.5 seconds
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 71.8MB/s]



Architecture: MobileNetV2
Training feature shape: (674, 1280)
Held-out raccoons accepted: 131 / 169
Raccoon acceptance rate: 77.51%
Time: 47.0 seconds


,Architecture,Raccoon acceptance rate (%),Time (seconds)
1,ResNet101,78.11,377.5
2,MobileNetV2,77.51,47.0
0,AlexNet,73.96,42.1


In [ ]:

def test_custom_image(model_name, img_path):
    model = build_feature_extractor(model_name)

    train_features = extract_features(model, train_loader)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)

    detector = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.1
    )
    detector.fit(train_scaled)

    img = Image.open(img_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        features = model(img_tensor).flatten(start_dim=1).cpu().numpy()

    features = scaler.transform(features)
    result = detector.predict(features)[0]

    if result == 1:
        print(model_name, ":", "Similar to learned raccoon images")
    else:
        print(model_name, ":", "Outlier relative to learned raccoon images")

img_path = "/content/raccoon.jpg"

for model_name in ["AlexNet", "ResNet101", "MobileNetV2"]:
    test_custom_image(model_name, img_path)

AlexNet : Outlier relative to learned raccoon images
ResNet101 : Outlier relative to learned raccoon images
MobileNetV2 : Outlier relative to learned raccoon images


In [ ]:
img_path = "/content/raccoon2.jpeg"

for model_name in ["AlexNet", "ResNet101", "MobileNetV2"]:
    test_custom_image(model_name, img_path)

AlexNet : Outlier relative to learned raccoon images
ResNet101 : Outlier relative to learned raccoon images
MobileNetV2 : Outlier relative to learned raccoon images


Need to fix the results here